# 04 Inventory Performance Analysis — 開發日誌

**資料來源：**
- Kaggle: kaggle.com/bhanupratapbiswas/inventory-analysis-case-study
- PwC 是 PricewaterhouseCoopers 的縮寫，全球四大會計師事務所（Big Four）之一
- 數據集的業務邏輯設計符合真實審計和財務分析標準，欄位設計反映的是真實企業的採購流程


---
## 📅 2026-04-14 — Phase 1：Raw Data 載入

### ✅ 完成事項
- 建立 `raw` / `staging` / `marts` 三層 Schema
- 執行 `sql/00_schema_setup.sql` 成功建立所有 raw tables
- 完成 `scripts/01_load_raw.py` Python Loader 腳本

---

### 🪲 踩坑紀錄 1：`00_schema_setup.sql` 出現 NOTICE 訊息

**現象：**
```
NOTICE: table "raw_sales" does not exist, skipping
Successfully run. Total query runtime: 113 msec.
1 rows affected.
```

**原因：**
- `NOTICE` 只是 PostgreSQL 的溫馨提示，不是錯誤（ERROR）
- SQL 中使用了 `DROP TABLE IF EXISTS`，第一次執行時找不到資料表，PostgreSQL 會提示 skipping
- `1 rows affected` 是最後一行 `SELECT 'Schema setup complete ✅'` 回傳了 1 行結果

**結論：** ✅ 完全正常，可放心繼續執行

---

### 🪲 踩坑紀錄 2：執行 Python 腳本時彈出新視窗、自動關閉

**現象：**
- 在 VSCode PowerShell 執行 `01_load_raw.py` 時，自動跳出一個新的黑色 Console 視窗
- 視窗跑完後自動關閉，看不到輸出結果

**原因：**
- Windows 環境下，某些情況下 Python 腳本會以獨立視窗模式啟動
- 程式執行結果無法保留在 VSCode 終端機中

**解決方法：**
- 直接在 VSCode 的終端機（PowerShell）輸入指令執行，而不是按 Run 按鈕
- 加上 `-u` 參數（Unbuffered）：
```bash
python -u scripts/01_load_raw.py
```

---

### 🪲 踩坑紀錄 3：`KeyboardInterrupt` 錯誤反覆出現

**現象：**
```
Traceback (most recent call last):
  File "scripts/01_load_raw.py", line 12, in <module>
    from sqlalchemy import create_engine, text
  ...
KeyboardInterrupt
```

**原因：**
- 這不是程式本身的 Bug！
- Python 在 Windows 上第一次載入 `pandas`、`sqlalchemy` 等大型套件時，需要花 **30 秒至 1 分鐘** 從硬碟讀取套件檔案
- 期間畫面黑黑的沒有任何反應，誤以為程式當機（Hang），於是按下 `Ctrl+C` 強制中止
- `Ctrl+C` 就會產生 `KeyboardInterrupt` 錯誤

**解決方法：**
- 執行指令後，把手離開鍵盤，**耐心等待至少 1 分鐘**
- 等到畫面出現以下文字才代表成功啟動：
```
=======================================================
04_Inventory_Performance_Analysis — Raw Loader
=======================================================
```

**教訓：** 沉住氣，看到進度文字出來才確認程式在跑 💡

---

### 📖 今日學習筆記

| 知識點 | 說明 |
|---|---|
| `python -u` | Unbuffered 模式，強制即時輸出 print 內容，不等暫存區滿才印出 |
| `DROP TABLE IF EXISTS` | PostgreSQL 找不到資料表時不報錯，只顯示 NOTICE |
| `load_dotenv()` | 從 `.env` 檔案讀取環境變數（資料庫密碼），避免密碼寫在程式碼中 |
| `os.getenv('KEY', 'fallback')` | 讀取環境變數，若找不到則使用預設值 |
| `encoding='utf-8-sig'` | 處理 CSV 的 BOM（\ufeff）問題，避免第一個欄位名稱被污染 |
| `dtype=str` | 讀取 CSV 時全部當作字串，避免 Pandas 自動型別推斷出錯 |
| 虛擬環境 (`04_env`) | 隔離專案套件，不影響電腦其他 Python 環境；用 `deactivate` 退出 |
